# **Step -1 : Apply the port5 feature extraction on the main sequence dataset and get the port5 feature extracted dataset**

In [ ]:
!pip install transformers torch sentencepiece

import pandas as pd
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
model = model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load your dataset
df = pd.read_csv("/content/dna_binding_dataset.csv")

# Validate sequence column
if "sequence" not in df.columns:
    raise ValueError("Expected column named 'sequence' in the dataset.")

# Function to embed sequence with ProtT5
def embed_sequence_prott5(seq):
    seq = " ".join(list(seq))  # add space between residues
    ids = tokenizer(seq, return_tensors="pt", padding=True)
    input_ids = ids['input_ids'].to(device)
    attention_mask = ids['attention_mask'].to(device)

    with torch.no_grad():
        embedding = model(input_ids=input_ids, attention_mask=attention_mask)

    # Take mean of encoder output (excluding padding)
    last_hidden = embedding.last_hidden_state.squeeze(0)
    mask = attention_mask.squeeze(0).unsqueeze(-1)
    mean_embedding = torch.sum(last_hidden * mask, dim=0) / torch.sum(mask)
    return mean_embedding.cpu().numpy()

# Apply embedding
features = []
for seq in tqdm(df['sequence'], desc="Embedding with ProtT5"):
    emb = embed_sequence_prott5(seq)
    features.append(emb)

# Create DataFrame
embedding_dim = len(features[0])
feature_cols = [f'ProtT5_{i+1}' for i in range(embedding_dim)]
prott5_df = pd.DataFrame(features, columns=feature_cols)

# Add label if available
if 'label' in df.columns:
    prott5_df['label'] = df['label']



# Save and download
prott5_df.to_csv("prott5_features_dataset.csv", index=False)

from google.colab import files
files.download("prott5_features_dataset.csv")


#**Step-2: Then apply 10 machine learning models for base line model and get a probability value dataset**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler

# === Load AAC features ===
aac_df = pd.read_csv('/content/prott5_protein_dataset.csv')  # Update with your AAC CSV path
X = aac_df.drop('label', axis=1).values
y = aac_df['label'].values

# === Models with best parameters ===
models = {
    "svc": SVC(
        C=17.585924233152554, kernel='rbf', gamma='scale', probability=True
    ),
    "knn": KNeighborsClassifier(
        n_neighbors=15, weights='distance', p=1
    ),
    "xgb": XGBClassifier(
        n_estimators=235, max_depth=10, learning_rate=0.0899272154665482,
        subsample=0.7577448093574966, colsample_bytree=0.8035372113305419,
        reg_alpha=0.5406130101658557, reg_lambda=0.748542082728425,
        eval_metric='logloss'   # ✅ removed use_label_encoder
    ),
    "rf": RandomForestClassifier(
        n_estimators=132, max_depth=14, min_samples_split=3, min_samples_leaf=1
    ),
    "et": ExtraTreesClassifier(
        n_estimators=175, max_depth=13, min_samples_split=6, min_samples_leaf=1,
        max_features=None, criterion='entropy'
    ),
    "gb": GradientBoostingClassifier(
        n_estimators=253, learning_rate=0.05684877327466396, max_depth=10,
        subsample=0.5625546581002723, min_samples_split=4, min_samples_leaf=5
    ),
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=12, min_samples_split=4, min_samples_leaf=2),
        n_estimators=100, max_samples=0.8, max_features=0.8, random_state=42
    ),
    "logreg": LogisticRegression(
        C=1.5, solver='liblinear', max_iter=500
    ),
    "dt": DecisionTreeClassifier(
        max_depth=12, min_samples_split=4, min_samples_leaf=2
    ),
    "ridge": RidgeClassifier(
        alpha=1.0, solver='auto', max_iter=500
    )
}

# === Prepare output dataframe ===
output_df = pd.DataFrame()
output_df["label"] = y

# === Stratified K-Fold for OOF predictions ===
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for short_name, model in models.items():
    print(f"Generating OOF predictions for {short_name}...")
    oof_preds = np.zeros(len(y))

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y[train_idx]

        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        else:
            try:
                scores = model.decision_function(X_val)
                oof_preds[val_idx] = MinMaxScaler().fit_transform(scores.reshape(-1, 1)).ravel()
            except:
                oof_preds[val_idx] = model.predict(X_val)

    output_df[f"attention_protein_{short_name}"] = oof_preds

# === Save results ===
output_path = "/content/merged_attention_protein_features_dataset_with_probs.csv"
output_df.to_csv(output_path, index=False)
print(f"✅ Leak-free OOF predictions saved to {output_path}")

# === Download CSV in Colab ===
from google.colab import files
files.download(output_path)


# **Step-3: Finally, apply RNN_LSTM (as a meta learner) on the probability value dataset and get the final result.**

In [ ]:
# Updated: Train & evaluate ONLY RNN-LSTM model (Best Performer for RBPs)
# -------------------------------------------------------------------
# Requirements: pandas, numpy, scikit-learn, tensorflow (2.x)
# -------------------------------------------------------------------

import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix
)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -------------------------
# Reproducibility
# -------------------------
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# Load dataset
# -------------------------
DATA_PATH = "/content/prott5_small_dataset_for_check.csv"
df = pd.read_csv(DATA_PATH)

TARGET_COL = "label"
X = df.drop(TARGET_COL, axis=1).values
y = df[TARGET_COL].values.astype(int)

# -------------------------
# Train-test split & scaling
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# RNN input reshaping (Timesteps = 1)
TIMESTEPS = 1
n_features = X_train.shape[1]
X_train_rnn = X_train.reshape((X_train.shape[0], TIMESTEPS, n_features))
X_test_rnn  = X_test.reshape((X_test.shape[0], TIMESTEPS, n_features))

# -------------------------
# Metrics function
# -------------------------
def compute_metrics(y_true, y_pred_labels, y_pred_probs):
    acc = accuracy_score(y_true, y_pred_labels)
    prec = precision_score(y_true, y_pred_labels, zero_division=0)
    rec = recall_score(y_true, y_pred_labels, zero_division=0)
    f1 = f1_score(y_true, y_pred_labels, zero_division=0)
    auc = roc_auc_score(y_true, y_pred_probs)
    kappa = cohen_kappa_score(y_true, y_pred_labels)
    mcc = matthews_corrcoef(y_true, y_pred_labels)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_labels).ravel()
    specificity = tn / (tn + fp)

    return {"Accuracy": acc, "Precision": prec, "Recall": rec,
            "F1": f1, "Specificity": specificity, "AUC": auc,
            "Kappa": kappa, "MCC": mcc}

# -------------------------
# Training utility
# -------------------------
def compile_and_train(model, X_tr, y_tr, epochs=120, batch_size=32, val_split=0.2):
    es = EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1)
    rl = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=5, verbose=1, min_lr=1e-6)
    history = model.fit(
        X_tr, y_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=val_split,
        callbacks=[es, rl],
        verbose=1
    )
    return history

# -------------------------
# RNN-LSTM Model Architecture (Optimized for RBP)
# -------------------------
def build_lstm(timesteps, features):
    model = Sequential([
        Input(shape=(timesteps, features)),
        LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    # Preserving the 1e-4 learning rate
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

# -------------------------
# Train RNN-LSTM Model
# -------------------------
print("\n===== Training RNN-LSTM (Best RBP Model) =====\n")

lstm_model = build_lstm(TIMESTEPS, n_features)
compile_and_train(lstm_model, X_train_rnn, y_train, epochs=120, batch_size=32)

# -------------------------
# Evaluate RNN-LSTM Model
# -------------------------
y_prob = lstm_model.predict(X_test_rnn).ravel()
y_pred = (y_prob > 0.5).astype(int)
metrics = compute_metrics(y_test, y_pred, y_prob)
metrics["Model"] = "RNN_LSTM"

results_df = pd.DataFrame([metrics])
results_df.to_csv("RNN_LSTM_prott5_results.csv", index=False)

print("\n===== Final RNN-LSTM Results =====\n")
print(results_df)

# **Visualization**

## Radar chart

In [ ]:
# ======= Colab-ready: DNA Binding Protein Train vs Test AAC Radar =======
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

# ---------- USER INPUT ----------
TRAIN_CSV = "/content/dna_binding_train_dataset.csv"   # Path to your training dataset
TEST_CSV  = "/content/dna_binding_test_dataset.csv"    # Path to your test dataset

# --- UPDATED COLORS ---
COLOR_BLUE = '#1f77b4'   # Assigned to DNAbindingProtein
COLOR_ORANGE = '#ff7f0e' # Assigned to Non_DNAbindingProtein

# Figure size (width, height)
FIGSIZE = (10, 5)

# ---------- FONT CONTROL ----------
plt.rcParams.update({
    "font.size": 12,
    'font.family': 'serif',
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})

# ---------- HELPERS ----------
AA20 = list("ACDEFGHIKLMNPQRSTVWY")

def _normalize(df):
    """Normalize columns: cleans sequence and handles 1/0 numeric labels."""
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    seq_col = next((col for col in df.columns if col in ['seq', 'sequence']), None)
    label_col = next((col for col in df.columns if col == 'label'), None)

    if seq_col is None or label_col is None:
         raise ValueError("CSV must contain identifiable 'sequence' and 'label' columns.")

    df.rename(columns={seq_col: "Sequence", label_col: "Label"}, inplace=True)
    df["Sequence"] = df["Sequence"].astype(str).str.upper().apply(lambda x: re.sub(r"[^A-Z]", "", x))
    df["Target"] = pd.to_numeric(df["Label"], errors='coerce')
    df = df.dropna(subset=["Target"])
    df["label_str"] = df["Target"].map({1: "D", 0: "ND"})
    return df

def _aac(seq):
    """Calculates Amino Acid Composition (AAC)"""
    n = len(seq) or 1
    counts = Counter(a for a in seq if a in AA20)
    return {f"AAC_{a}": counts.get(a, 0)/n for a in AA20}

def _avg_table(df):
    """Computes average AAC for DNAbinding vs Non-DNAbinding classes."""
    aac_data = []
    for _, row in df.iterrows():
        aac_res = _aac(row.Sequence)
        aac_res['label'] = row.label_str
        aac_data.append(aac_res)

    feat = pd.DataFrame(aac_data)
    for aa in AA20:
        col = f"AAC_{aa}"
        if col not in feat.columns:
            feat[col] = 0.0

    sorted_aac_cols = [f"AAC_{aa}" for aa in AA20]
    return pd.DataFrame({
        "AA": AA20,
        "D_percent": feat[feat["label"] == "D"][sorted_aac_cols].mean().values * 100,
        "ND_percent": feat[feat["label"] == "ND"][sorted_aac_cols].mean().values * 100
    })

def _radar(ax, labels, vD, vN, title):
    """Plots the radar chart with class-specific colors."""
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]
    vD = list(vD) + [vD[0]]
    vN = list(vN) + [vN[0]]

    ax.plot(angles, vD, linewidth=2, label="DNAbindingProtein", color=COLOR_BLUE)
    ax.fill(angles, vD, alpha=0.25, color=COLOR_BLUE)
    ax.plot(angles, vN, linewidth=2, label="Non_DNAbindingProtein", color=COLOR_ORANGE)
    ax.fill(angles, vN, alpha=0.25, color=COLOR_ORANGE)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(title, pad=30)
    ax.yaxis.grid(True, linestyle="--", linewidth=1.2, alpha=0.7)
    ax.xaxis.grid(True, linestyle="--", linewidth=1.2, alpha=0.7)

# ---------- MAIN EXECUTION ----------
try:
    df_train = _normalize(pd.read_csv(TRAIN_CSV))
    df_test  = _normalize(pd.read_csv(TEST_CSV))

    train_avg = _avg_table(df_train)
    test_avg  = _avg_table(df_test)

    labels = train_avg["AA"].tolist()
    vD_tr, vN_tr = train_avg["D_percent"], train_avg["ND_percent"]
    vD_te, vN_te = test_avg["D_percent"], test_avg["ND_percent"]

    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, subplot_kw=dict(polar=True), gridspec_kw=dict(wspace=0.4))

    # Plot A: Training Set (Legend anchored here to appear in the center)
    _radar(axes[0], labels, vD_tr, vN_tr, "(A) Training Set")
    axes[0].legend(loc="upper center", bbox_to_anchor=(1.2, 1.15), frameon=False, ncol=2)

    # Plot B: Test Set (No legend)
    _radar(axes[1], labels, vD_te, vN_te, "(B) Test Set")

    plt.tight_layout()
    output_filename = "Train_vs_Test_aac_radar_single_legend.pdf"
    plt.savefig(output_filename, dpi=900, bbox_inches="tight")
    print(f"Successfully generated side-by-side radar chart: {output_filename}")
    plt.show()

except Exception as e:
    print(f"Error encountered: {e}")

# **Shap**

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import files

# 1. Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. Load Dataset
# Note: This dataset contains the 10 probability columns from your base models
DATA_PATH = "/content/ProtT5_features_dataset_with_probs.csv"
df = pd.read_csv(DATA_PATH)

TARGET_COL = "label"
X = df.drop(TARGET_COL, axis=1)
feature_names = X.columns.tolist() # Captures "attention_protein_svc", etc.
y = df[TARGET_COL].values.astype(int)

# 3. Data Preparation
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.20, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for RNN-LSTM (Samples, Timesteps=1, Features=10)
TIMESTEPS = 1
n_features = X_train_scaled.shape[1]
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], TIMESTEPS, n_features))
X_test_rnn  = X_test_scaled.reshape((X_test_scaled.shape[0], TIMESTEPS, n_features))

# 4. Build and Train RNN-LSTM Meta-Model
def build_lstm(timesteps, features):
    model = Sequential([
        Input(shape=(timesteps, features)),
        LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("\n===== Training RNN-LSTM Meta-Model =====\n")
lstm_model = build_lstm(TIMESTEPS, n_features)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=5)
]

lstm_model.fit(X_train_rnn, y_train, epochs=120, batch_size=32,
               validation_split=0.2, callbacks=callbacks, verbose=1)

# 5. Interpretation Stage: SHAP Visualization
# We import SHAP after training to ensure the TF gradient registry remains clean
import shap

print("\n[INFO] Generating SHAP Explanations (Beeswarm Plot)...")

# SHAP wrapper: Reshapes 2D input from SHAP back to 3D for the LSTM
def model_predict_wrapper(x_2d):
    x_3d = x_2d.reshape((x_2d.shape[0], TIMESTEPS, n_features))
    return lstm_model.predict(x_3d, verbose=0).flatten()

# Use a background summary (50 samples) for faster calculation
background_summary = shap.sample(X_train_scaled, 50)
explainer = shap.KernelExplainer(model_predict_wrapper, background_summary)

# Calculate SHAP values for a subset of the test data
X_test_sample = X_test_scaled[:100]
shap_values = explainer.shap_values(X_test_sample)

# 6. Visualization and Export
plt.figure(figsize=(12, 8))



shap.summary_plot(
    shap_values,
    X_test_sample,
    feature_names=feature_names,
    plot_type="dot", # Creates the "Beeswarm" effect
    show=False
)

plt.title("", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("SHAP Value (Impact on RNA-binding Probability)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()

# Save for publication
plt.savefig("RBP_SHAP_Analysis.png", dpi=900, bbox_inches='tight')
plt.savefig("RBP_SHAP_Analysis.pdf", bbox_inches='tight')
files.download("RBP_SHAP_Analysis.png")
print("\n[SUCCESS] SHAP Beeswarm plot saved and downloaded.")
plt.show()